# LLM Scaling: Multi-Head Latent Attention

## Multi-Head Latent Attention (MLA)

## The KV-Cache Problem

During autoregressive generation, we cache **Keys and Values** to avoid
recomputing them at each step.  For standard MHA the KV-cache is:

$$\text{KV cache size} = 2 \times n_{heads} \times d_{head} \times L_{seq} \times \text{bytes}$$

For **DeepSeek-V2** (128 heads, 128 d_head, 128k context, fp16):
$2 \times 128 \times 128 \times 128000 \times 2 \approx 8.4\text{ GB per request}$.

**Multi-Head Latent Attention** (MLA) compresses this dramatically.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ── Standard Multi-Head Attention with explicit KV-cache ─────────────────────
class StandardMHA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head    = d_model // num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        B, T, D = x.shape
        H, Dh   = self.num_heads, self.d_head

        Q = self.Wq(x).reshape(B, T, H, Dh).transpose(1, 2)  # (B, H, T, Dh)
        K = self.Wk(x).reshape(B, T, H, Dh).transpose(1, 2)
        V = self.Wv(x).reshape(B, T, H, Dh).transpose(1, 2)

        # Extend KV cache for incremental decoding
        if kv_cache is not None:
            K = torch.cat([kv_cache[0], K], dim=2)
            V = torch.cat([kv_cache[1], V], dim=2)
        new_cache = (K, V)

        scale = Dh ** -0.5
        attn  = torch.softmax(Q @ K.transpose(-2, -1) * scale, dim=-1)
        out   = (attn @ V).transpose(1, 2).reshape(B, T, D)
        return self.Wo(out), new_cache

    def kv_cache_bytes(self, seq_len, bytes_per_element=2):
        """KV cache size in bytes for a given sequence length."""
        return 2 * self.num_heads * self.d_head * seq_len * bytes_per_element


# Test
mha = StandardMHA(d_model=512, num_heads=8)
x   = torch.randn(1, 16, 512)
out, cache = mha(x)
print(f"MHA output: {out.shape}")
print(f"KV cache shapes: K={cache[0].shape}, V={cache[1].shape}")
print(f"KV cache at seq=1024: {mha.kv_cache_bytes(1024)/1024:.1f} KB")
print(f"KV cache at seq=32k : {mha.kv_cache_bytes(32768)/1024/1024:.2f} MB")


## Multi-Head Latent Attention (MLA) — DeepSeek-V2/V3

MLA compresses Keys and Values through a **low-rank bottleneck**,
caching only the compressed latent vector $c_{KV}$.

### Architecture

$$c_{KV} = W_{DKV} \, x \in \mathbb{R}^{r}  \quad (r \ll n_h \cdot d_h)$$
$$K = W_{UK} \, c_{KV}, \quad V = W_{UV} \, c_{KV}$$

At inference time, only $c_{KV}$ (size $r$) is cached instead of full K and V
(size $2 \times n_h \times d_h$).

MLA also introduces **decoupled RoPE**: a separate positional key $k_R$
computed from the raw input (not from the compressed latent), which allows
RoPE to be applied correctly even with latent compression.


In [ ]:
def apply_rotary_pos_emb(x: torch.Tensor) -> torch.Tensor:
    """Applies 2D rotary position embeddings (RoPE) to the last dimension."""
    # x shape: (B, H, T, D_r)
    B, H, T, D = x.shape
    half_d = D // 2
    inv_freq = 1.0 / (10000.0 ** (torch.arange(0, half_d, device=x.device).float() / half_d))
    t = torch.arange(T, device=x.device).float()
    freqs = torch.outer(t, inv_freq)  # (T, half_d)
    cos = freqs.cos().unsqueeze(0).unsqueeze(1)  # (1, 1, T, half_d)
    sin = freqs.sin().unsqueeze(0).unsqueeze(1)
    
    x1, x2 = x[..., :half_d], x[..., half_d:]
    rx1 = x1 * cos - x2 * sin
    rx2 = x1 * sin + x2 * cos
    return torch.cat([rx1, rx2], dim=-1)

class MultiHeadLatentAttention(nn.Module):
    """
    Multi-Head Latent Attention (MLA) with Decoupled RoPE as in DeepSeek-V2/V3.

    Parameters
    ----------
    d_model        : token embedding dimension
    num_heads      : number of attention heads
    kv_lora_rank   : dimension of compressed KV latent (r_kv)
    q_lora_rank    : dimension of compressed Q latent (r_q), or None for direct Q
    rope_head_dim  : per-head dimension for decoupled rotary keys and queries
    """
    def __init__(self, d_model: int, num_heads: int, kv_lora_rank: int = 64, q_lora_rank: int = None, rope_head_dim: int = 32):
        super().__init__()
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.kv_lora_rank = kv_lora_rank
        self.rope_head_dim = rope_head_dim

        # ── 1. Query Projections ──────────────────────────────────────────────
        if q_lora_rank is not None:
            self.Wdq = nn.Linear(d_model, q_lora_rank, bias=False)
            self.Wuq = nn.Linear(q_lora_rank, num_heads * self.d_head, bias=False)
        else:
            self.Wdq = None
            self.Wuq = nn.Linear(d_model, num_heads * self.d_head, bias=False)
        # Query RoPE projection (decoupled)
        self.Wqr = nn.Linear(d_model, num_heads * rope_head_dim, bias=False)

        # ── 2. Key/Value Projections & Compression (Core Innovation) ──────────
        # Down-project KV into low-rank latent vector c_kv (THIS IS WHAT GETS CACHED)
        self.Wdkv = nn.Linear(d_model, kv_lora_rank, bias=False)
        # Up-project latent to content keys and values
        self.Wuk = nn.Linear(kv_lora_rank, num_heads * self.d_head, bias=False)
        self.Wuv = nn.Linear(kv_lora_rank, num_heads * self.d_head, bias=False)

        # Decoupled Key RoPE projection (computed directly from x, cached alongside c_kv)
        self.Wkr = nn.Linear(d_model, rope_head_dim, bias=False)  # shared RoPE key across heads or per head

        # ── 3. Output Projection ──────────────────────────────────────────────
        self.Wo = nn.Linear(num_heads * self.d_head, d_model, bias=False)

    def forward(self, x: torch.Tensor, past_kv_cache: tuple = None):
        """
        x: (B, T, D)
        past_kv_cache: tuple (cached_c_kv, cached_k_pe) or None
        """
        B, T, D = x.shape
        H, Dh, Dr = self.num_heads, self.d_head, self.rope_head_dim

        # ── Queries ──────────────────────────────────────────────────────────
        q_c = self.Wuq(self.Wdq(x) if self.Wdq is not None else x).reshape(B, T, H, Dh).transpose(1, 2)
        q_pe = self.Wqr(x).reshape(B, T, H, Dr).transpose(1, 2)
        q_pe = apply_rotary_pos_emb(q_pe)  # apply RoPE to decoupled query

        # ── KV Compression (c_kv) & Decoupled RoPE Key (k_pe) ────────────────
        c_kv = self.Wdkv(x)  # (B, T, r_kv) - compressed content latent
        k_pe = self.Wkr(x).unsqueeze(1)  # (B, 1, T, Dr)
        k_pe = apply_rotary_pos_emb(k_pe.expand(B, H, T, Dr))  # (B, H, T, Dr)

        # Incremental KV-cache concatenation
        if past_kv_cache is not None:
            prev_c_kv, prev_k_pe = past_kv_cache
            c_kv = torch.cat([prev_c_kv, c_kv], dim=1)
            k_pe = torch.cat([prev_k_pe, k_pe], dim=2)
        new_cache = (c_kv, k_pe)

        # Up-project cached latent to content Keys and Values
        k_c = self.Wuk(c_kv).reshape(B, -1, H, Dh).transpose(1, 2)  # (B, H, T_kv, Dh)
        v_c = self.Wuv(c_kv).reshape(B, -1, H, Dh).transpose(1, 2)  # (B, H, T_kv, Dh)

        # ── Attention with Content + Position Decomposition ──────────────────
        # Attention score: A = (q_c @ k_c.T) + (q_pe @ k_pe.T)
        scale = (Dh + Dr) ** -0.5
        attn_scores = (q_c @ k_c.transpose(-2, -1) + q_pe @ k_pe.transpose(-2, -1)) * scale
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        out = (attn_probs @ v_c).transpose(1, 2).reshape(B, T, H * Dh)
        return self.Wo(out), new_cache

    def kv_cache_elements_per_token(self) -> int:
        # DeepSeek MLA caches: compressed latent c_kv (r_kv) + decoupled RoPE key (Dr)
        return self.kv_lora_rank + self.rope_head_dim

# ── Instantiate and test ─────────────────────────────────────────────────────
d_model, num_heads = 512, 8
mla = MultiHeadLatentAttention(d_model=d_model, num_heads=num_heads, kv_lora_rank=64, q_lora_rank=128, rope_head_dim=32)
mha = StandardMHA(d_model=d_model, num_heads=num_heads)

x = torch.randn(1, 16, d_model)
mla_out, mla_cache = mla(x)
mha_out, kv_cache = mha(x)

print(f'MLA output shape : {mla_out.shape}')
print(f'MHA output shape : {mha_out.shape}')
print(f'MLA cached c_kv  : {mla_cache[0].shape}  (r_kv=64)')
print(f'MLA cached k_pe  : {mla_cache[1].shape}  (Dr=32)')
print(f'MHA K cache shape: {kv_cache[0].shape}  (H=8, Dh=64)')
print(f'MHA V cache shape: {kv_cache[1].shape}  (H=8, Dh=64)')
print(f'MLA Cache elements / token: {mla.kv_cache_elements_per_token()} vs MHA: {2 * d_model} ({2*d_model / mla.kv_cache_elements_per_token():.1f}x reduction!)')


In [ ]:
# ── KV cache size comparison across attention variants ───────────────────────
seq_lengths = [1024, 4096, 16384, 32768, 65536, 131072]

d_model_full  = 4096      # realistic LLM size
num_heads_gqa = 8         # GQA heads
num_heads_mha = 32
kv_rank_mla   = 512       # DeepSeek-V2 uses 512

# MHA KV cache = 2 * H * Dh * T * 2 bytes
mha_bytes  = [2 * num_heads_mha  * (d_model_full // num_heads_mha) * T * 2
              for T in seq_lengths]

# GQA (Grouped Query Attention) = 2 * H_kv * Dh * T * 2 bytes
gqa_bytes  = [2 * num_heads_gqa  * (d_model_full // num_heads_mha) * T * 2
              for T in seq_lengths]

# MLA = r_kv * T * 2 bytes  (only compressed latent cached)
mla_bytes  = [kv_rank_mla * T * 2 for T in seq_lengths]

def to_gb(b): return b / 1e9

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([T/1000 for T in seq_lengths], [to_gb(b) for b in mha_bytes],
        "o-", label=f"MHA ({num_heads_mha} heads)", linewidth=2)
ax.plot([T/1000 for T in seq_lengths], [to_gb(b) for b in gqa_bytes],
        "s--", label=f"GQA ({num_heads_gqa} KV heads)", linewidth=2)
ax.plot([T/1000 for T in seq_lengths], [to_gb(b) for b in mla_bytes],
        "^-", label=f"MLA (r_kv={kv_rank_mla})", linewidth=2, color="green")

ax.set(xlabel="Sequence length (k tokens)", ylabel="KV-cache size (GB)",
       title=f"KV-cache Size vs Sequence Length\n(d_model={d_model_full}, fp16, one layer)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Print ratio at max sequence length
T = seq_lengths[-1]
print(f"\nAt seq_len = {T:,} tokens (one transformer layer, fp16):")
print(f"  MHA  KV cache : {to_gb(mha_bytes[-1]):.2f} GB")
print(f"  GQA  KV cache : {to_gb(gqa_bytes[-1]):.2f} GB  ({mha_bytes[-1]/gqa_bytes[-1]:.0f}× smaller than MHA)")
print(f"  MLA  KV cache : {to_gb(mla_bytes[-1]):.3f} GB  ({mha_bytes[-1]/mla_bytes[-1]:.0f}× smaller than MHA)")
print(f"\nAcross all {num_heads_mha * 2} transformer layers (typical LLM):")
print(f"  MHA  total : {to_gb(mha_bytes[-1]) * num_heads_mha * 2:.1f} GB")
print(f"  MLA  total : {to_gb(mla_bytes[-1]) * num_heads_mha * 2:.2f} GB")


## Summary

### Multi-Head Latent Attention

| Method | What is cached | Cache reduction vs MHA |
|---|---|---|
| MHA | Full K, V for all heads | 1× |
| GQA | K, V for fewer KV heads | 4–8× |
| MLA | Compressed latent $c_{KV}$ | 10–100× |

**MLA trade-off**: lower memory cost at inference, but the up-projection $W_{UK}, W_{UV}$
must be applied at every decoding step — additional compute that is typically offset by
batching more requests per GPU.
